# 2. Nettoyage et valeurs manquantes

Code dans `src/cleaning.py`. Étapes : codes invalides et bornes, jours sans montre, valeurs extrêmes, puis comparaison de plusieurs façons de remplir les valeurs manquantes.

In [1]:
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:   # Colab : code depuis GitHub, données depuis Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = "/content/drive/MyDrive/Colab Notebooks/athlete-data/data/raw"
    repo = Path("/content/athlete-monitoring")
    if repo.exists():
        for generated in ("notebooks", "data/processed"):   # fichiers régénérés, à remettre avant la mise à jour
            subprocess.run(["git", "-C", str(repo), "checkout", "--", generated], capture_output=True)
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Harena21/athlete-monitoring.git", str(repo)],
                       check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(repo)
sys.path.insert(0, str(repo))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
COLORS = {"p01": "#2a78d6", "p03": "#eb6834", "p05": "#1baf7a"}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from src.data import build_daily
from src import cleaning
daily = build_daily()
clean = cleaning.clean(daily)

## Codes invalides, bornes, jours sans montre

Un 0 sur une échelle de 1 à 5 ou une readiness à 0 devient une valeur manquante, comme une valeur impossible (FC de repos hors de 30 à 120, par exemple). Un jour sans montre (moins de 10 h de FC enregistrée, ou moins de 200 pas quand il n'y a pas de FC) n'est pas compté comme un jour de repos : son activité passe à NaN, sinon la charge serait sous-estimée.

In [3]:
changed = {c: int((daily[c].notna() & clean[c].isna()).sum()) for c in ["well_fatigue", "well_readiness", "steps", "resting_hr"]}
print("valeurs passées à NaN :", changed)
print("jours sans montre :", clean.groupby("participant")["no_watch"].sum().to_dict())

valeurs passées à NaN : {'well_fatigue': 2, 'well_readiness': 57, 'steps': 19, 'resting_hr': 0}
jours sans montre : {'p01': 0, 'p03': 44, 'p05': 12}


## Valeurs extrêmes

z-score robuste (médiane et MAD) par athlète, seuil 3,5 (Iglewicz et Hoaglin, 1993). Je les signale dans une colonne `_outlier` sans les supprimer : un vrai jour d'effort très intense reste une information.

In [4]:
clean.groupby("participant")[["steps_outlier", "trimp_outlier", "sleep_minutes_outlier"]].sum()

,steps_outlier,trimp_outlier,sleep_minutes_outlier
participant,,,
p01,0,4,2
p03,0,4,3
p05,0,5,3


## Valeurs manquantes : quelle méthode ?

Je cache 10 % des valeurs connues, je les remplis avec chaque méthode et je mesure l'erreur (RMSE divisée par l'écart-type de la variable, 1 correspondant à peu près au niveau de la moyenne). Je répète avec trois tirages différents.

In [5]:
cols = ["steps", "resting_hr", "trimp", "sleep_minutes", "sleep_score", "sleep_efficiency",
        "well_readiness", "well_fatigue", "well_mood", "well_stress", "weight"]
res = pd.concat({f"tirage {seed}": cleaning.compare_imputers(clean, cols, seed=seed) for seed in (42, 1, 2)}, axis=1)
res["moyenne des tirages"] = res.mean(axis=1)
res.sort_values("moyenne des tirages").round(3)

,tirage 42,tirage 1,tirage 2,moyenne des tirages
MICE,0.936,0.787,0.986,0.903
interpolation,0.853,0.961,0.973,0.929
médiane de l'athlète,0.927,0.970,1.042,0.980
KNN,0.922,1.030,1.112,1.021
moyenne,1.011,1.057,1.145,1.071


MICE (régressions en chaîne avec des forêts aléatoires) a la plus faible erreur en moyenne sur les trois tirages, juste devant l'interpolation. L'ordre change d'un tirage à l'autre, l'écart entre les deux est donc faible. Le KNN reste proche de la moyenne : au premier essai il faisait même pire, parce que les pas (des milliers) écrasaient les échelles de 1 à 5 dans le calcul des distances, et je l'ai corrigé en standardisant.

Pour le jeu livré, j'utilise donc MICE, avec quelques garde-fous :
- je ne remplis pas une variable presque toujours absente chez un athlète (plus de 80 %, comme la FC de p03) ;
- les réponses au questionnaire restent arrondies dans leur échelle ;
- je ne remplis ni les blessures ni la sRPE (sans réponse on ne sait pas s'il y a eu blessure ou séance), ni les calories des photos (février et mars seulement), ni l'indice de performance qui sert de cible.

Pour les modèles, le remplissage est refait dans chaque pli de validation, pour ne pas utiliser d'information du jeu de test.

## Export de test_dataset.csv

In [6]:
from main import build_dataset
build_dataset()
test = pd.read_csv("data/processed/test_dataset.csv")
print(test.shape)
test.head()

(456, 41)


,date,steps,calories,distance_km,hr_mean,hr_wear_minutes,resting_hr,zone1_min,zone2_min,zone3_min,...,monotony,resting_hr_delta,no_watch,steps_outlier,trimp_outlier,sleep_minutes_outlier,perf_index,food_n_photos,food_kcal,wellness_missing
0,2019-11-01,17873.0,4009.10,14.42400,66.140075,1421.0,53.741074,164.0,3.0,0.0,...,2.410663,0.740416,0,0,0,0,33.831949,NaN,NaN,0
1,2019-11-02,13118.0,3533.56,10.58480,62.602226,1393.0,52.881497,137.0,0.0,0.0,...,2.023390,0.270948,0,0,0,0,56.770957,NaN,NaN,0
2,2019-11-03,14312.0,3748.73,11.46085,64.551841,1368.0,53.222024,141.0,0.0,0.0,...,2.709823,0.617956,0,0,0,0,68.910866,NaN,NaN,0
3,2019-11-04,10970.0,3353.38,8.85970,63.552109,1414.0,54.311141,100.0,0.0,0.0,...,4.716198,1.895792,0,0,0,0,38.594112,NaN,NaN,0
4,2019-11-05,16186.0,3794.63,13.71470,66.672530,1412.0,52.259110,94.0,7.0,12.0,...,5.405852,-0.633710,0,0,0,0,74.237925,NaN,NaN,0
